## 데이터 전처리와 파이프라인


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

SEED = 42


### 전처리가 필요한 이유

전처리는 수집한 데이터를 모델이 사용할 수 있는 입력으로 정리하고 변환하는 작업이다.
빠진 값이나 잘못된 기록을 처리하고, 문자열과 서로 다른 단위의 숫자를 모델에 맞게 표현한다.

| 데이터의 상태 | 필요한 처리 |
| --- | --- |
| 같은 기록이 반복되거나 표기·단위가 섞임 | 중복 확인, 표기·단위 통일 |
| 값이 비어 있거나 잘못 기록됨 | 오류 확인, 삭제 또는 결측값 대체 |
| 숫자 열마다 단위와 크기가 다름 | 표준화 등 스케일링 |
| 섬 이름처럼 종류를 나타내는 문자열이 있음 | 원핫 인코딩 등 범주 표현 |

전처리 방법은 데이터의 상태와 처리 후 남는 정보를 고려해 선택한다. 결측값이 있는 행을 삭제할 때는 결측 비율과 남는 데이터의 양을 확인하고, 유난히 큰 값은 입력 오류인지 실제 관측값인지 살펴본다. 스케일링은 거리 기반 모델처럼 숫자 크기에 영향을 받는 모델에서 고려한다.
데이터에서 구한 대체값과 변환 기준은 학습 데이터로 정하고, 검증·테스트 데이터와 새 데이터에 그대로 적용한다.

### 펭귄 데이터와 예측 문제

몸무게·날개 길이·관측한 섬으로 펭귄 종류를 예측하는 분류 문제의 입력을 준비한다.
Penguins는 Palmer Penguins 관측 자료를 담은 seaborn 예제 데이터다. 한 행은 펭귄 한 마리이며, 전체 344행이다.

| 입력 열 | 뜻 | 자료형과 단위 |
| --- | --- | --- |
| `body_mass_g` | 몸무게 | 숫자, g |
| `flipper_length_mm` | 날개 길이 | 숫자, mm |
| `island` | 관측한 섬 | 문자, `Biscoe`·`Dream`·`Torgersen` |

`species`는 예측할 펭귄 종류로 `Adelie`, `Chinstrap`, `Gentoo`가 있다.

`sns.load_dataset("penguins")`는 예제 데이터를 행과 열로 구성된 DataFrame으로 불러온다.


In [ ]:
# 펭귄 데이터를 불러오고 전체 크기와 일부 행 확인
penguins = sns.load_dataset("penguins")
print("전체 행과 열:", penguins.shape)
display(penguins.head())


## 모델 입력 준비


### 입력 선택과 결측값 확인

펭귄 데이터에서 숫자 두 열과 범주 한 열을 입력 `X`로 선택한다.

`isna()`는 각 값이 결측값이면 `True`, 아니면 `False`로 표시한다. 처음 5행에서 원래 값과 결측 여부를 비교한다.
`sum()`은 기본으로 열별 합계를 구한다. `True`를 1, `False`를 0으로 세므로 `X.isna().sum()`은 각 열의 결측값 개수를 반환한다.

`any(axis=1)`은 각 행에 `True`가 하나라도 있는지 확인한다. `X[X.isna().any(axis=1)]`은 결측값이 하나라도 있는 행만 선택한다.


In [ ]:
# 숫자 열과 범주 열을 나누어 입력 선택
numeric_columns = ["body_mass_g", "flipper_length_mm"]
category_columns = ["island"]
X = penguins[numeric_columns + category_columns].copy()

# 처음 5행의 원래 값과 결측 여부 비교
display(X.head())
display(X.isna().head())

# 전체 데이터의 열별 결측값 수 확인
display(X.isna().sum())

# 결측값이 있는 관측 기록 확인
display(X[X.isna().any(axis=1)])


### 학습·검증·테스트로 나누기

학습 데이터에서 전처리 기준을 구하고, 검증·테스트 데이터에는 그 기준을 적용한다.
검증·테스트 데이터까지 섞어서 평균이나 중앙값을 구하면, 평가에 쓸 데이터의 정보가 학습 과정에 들어가는 데이터 누수가 생긴다.

| 구분 | 역할 | 전처리 |
| --- | --- | --- |
| 학습 | 모델과 전처리 기준 학습 | 기준을 구하고 변환 |
| 검증 | 모델·전처리 설정 선택 | 학습 기준으로 변환 |
| 테스트 | 선택을 마친 모델의 최종 평가 | 학습 기준으로 변환 |

학습 60%, 검증 20%, 테스트 20%로 나눈다. 이 실습에서는 세 데이터의 전처리 결과를 확인한다.

`train_test_split`은 입력의 행을 무작위로 나눈다. 먼저 `test_size=0.2`로 테스트 데이터를 분리하고, 남은 80%에서 `test_size=0.25`로 전체의 20%를 검증용으로 분리한다.
`random_state=SEED`는 같은 무작위 분리를 재현하는 설정이다.


In [ ]:
# 전체 데이터에서 테스트용 20% 분리
X_train_val, X_test = train_test_split(
    X, test_size=0.2, random_state=SEED,
)

# 남은 데이터에서 검증용을 분리해 학습 60%·검증 20%로 구성
X_train, X_val = train_test_split(
    X_train_val, test_size=0.25, random_state=SEED,
)
print("학습 데이터:", X_train.shape)
print("검증 데이터:", X_val.shape)
print("테스트 데이터:", X_test.shape)


## 개별 전처리


### 전처리 도구의 공통 사용법

전처리 도구는 학습 데이터에서 기준을 구해 객체 안에 저장한다.
같은 객체로 검증·테스트 데이터와 새 데이터를 변환하면 학습할 때 정한 기준이 유지된다.

| 메서드 | 역할 | 결측값 대체 예시 |
| --- | --- | --- |
| `fit` | 기준 계산·저장 | 학습 데이터의 열별 중앙값 저장 |
| `transform` | 저장한 기준으로 변환 | 저장된 중앙값으로 결측값 대체 |
| `fit_transform` | 기준 계산·저장 후 변환 | 중앙값 저장 후 학습 결측값 대체 |


### 중앙값으로 결측값 채우기

`SimpleImputer`는 열별로 정한 대표값이나 고정값으로 결측값을 채우는 전처리 클래스다.
중앙값은 값을 크기순으로 정렬했을 때 가운데 값이며, 개수가 짝수이면 가운데 두 값의 평균이다. 평균보다 극단값의 영향을 덜 받는다.

몸무게와 날개 길이의 결측값을 학습 데이터에서 구한 각 열의 중앙값으로 채울 수 있다.

`strategy="median"`은 중앙값 대체를 지정한다. `fit_transform`은 학습 중앙값을 저장하고 학습 데이터를 변환하며, `transform`은 저장된 중앙값으로 검증·테스트 데이터를 변환한다.
반환값은 NumPy 배열로, 학습 결과는 `(206, 2)`, 검증·테스트 결과는 각각 `(69, 2)`다.

`imputer.statistics_`에서 입력 열 순서로 저장된 대체값을 확인할 수 있다.


In [ ]:
# 학습 데이터의 중앙값으로 학습·검증·테스트 데이터의 빈칸 채우기
imputer = SimpleImputer(strategy="median")
train_filled = imputer.fit_transform(X_train[numeric_columns])
val_filled = imputer.transform(X_val[numeric_columns])
test_filled = imputer.transform(X_test[numeric_columns])

# 저장된 대체값 확인
display(pd.Series(imputer.statistics_, index=numeric_columns, name="학습 중앙값"))

# 몸무게가 비어 있던 행의 대체 전후 비교
train_filled_df = pd.DataFrame(train_filled, columns=numeric_columns, index=X_train.index)
missing_rows = X_train["body_mass_g"].isna()
print("빈칸을 채우기 전")
display(X_train.loc[missing_rows, numeric_columns])
print("빈칸을 채운 후")
display(train_filled_df.loc[missing_rows, numeric_columns])


### 숫자의 크기를 맞추는 표준화

몸무게의 4,000g과 날개 길이의 200mm는 단위와 숫자의 크기가 다르다.
거리 기반 모델에서는 숫자 크기가 큰 열이 거리 계산에 더 큰 영향을 줄 수 있다.
표준화는 각 값을 평균에서 표준편차 몇 개만큼 떨어져 있는지로 바꾼다.

```text
변환값 = (원래 값 - 평균) / 표준편차
```

표준편차는 값들이 평균 주변에 얼마나 퍼져 있는지를 나타낸다.
`StandardScaler`는 학습 데이터의 열별 평균과 표준편차를 저장하고, 그 기준으로 표준화하는 전처리 클래스다.

`scaler.mean_`에는 학습 데이터의 열별 평균이, `scaler.scale_`에는 표준화에 사용한 열별 표준편차가 저장된다.


In [ ]:
# 학습 데이터에서 구한 기준으로 학습·검증·테스트 데이터 표준화
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_filled)
val_scaled = scaler.transform(val_filled)
test_scaled = scaler.transform(test_filled)

# 표준화 기준과 변환 전후 값 확인
display(pd.DataFrame({"평균": scaler.mean_, "표준편차": scaler.scale_},
                     index=numeric_columns))
print("표준화 전")
display(train_filled_df.head())
print("표준화 후")
display(pd.DataFrame(train_scaled, columns=numeric_columns, index=X_train.index).head())


### 표준화 결과 읽기

변환값이 음수이면 원래 측정값이 학습 평균보다 작다는 뜻이다. 0은 학습 평균과 같고, 1은 학습 평균보다 표준편차 한 개만큼 크다는 뜻이다.

변환한 학습 데이터의 열별 평균은 약 0, 표준편차는 약 1이다.
검증·테스트 데이터는 학습 데이터의 기준으로 변환하므로 평균이 0에서 벗어날 수 있다. 세 데이터의 평균을 비교한다.

NumPy 배열의 `mean()`은 평균, `std()`는 표준편차를 계산한다. `axis=0`으로 지정하면 각 열을 따로 계산해 1차원 배열로 반환한다.


In [ ]:
# 표준화 후 학습·검증·테스트 데이터의 열별 통계 비교
display(pd.DataFrame({
    "변환 후 학습 평균": train_scaled.mean(axis=0),
    "변환 후 학습 표준편차": train_scaled.std(axis=0),
    "변환 후 검증 평균": val_scaled.mean(axis=0),
    "변환 후 테스트 평균": test_scaled.mean(axis=0),
}, index=numeric_columns).round(4))


### 섬 이름을 숫자로 표현하기

섬 이름처럼 종류를 나타내는 값을 범주라고 하고, 범주를 숫자로 표현하는 작업을 인코딩이라고 한다.

원핫 인코딩은 범주마다 열을 만들고 해당하면 1, 아니면 0을 넣는 방법이다.
`OneHotEncoder`는 학습 데이터의 범주 목록을 저장하고, 같은 열 순서로 인코딩하는 전처리 클래스다.

| 섬 | Biscoe | Dream | Torgersen |
| --- | --- | --- | --- |
| Biscoe | 1 | 0 | 0 |
| Dream | 0 | 1 | 0 |
| Torgersen | 0 | 0 | 1 |

| 설정·메서드 | 역할 |
| --- | --- |
| `handle_unknown="ignore"` | 학습 때 없던 섬은 섬을 나타내는 열을 모두 0으로 처리 |
| `sparse_output=False` | 희소 행렬 대신 일반 NumPy 배열로 반환 |
| `get_feature_names_out()` | 입력 열과 범주 이름으로 만든 출력 열 이름 반환 |

섬 세 종류를 인코딩한 배열의 모양은 학습 `(206, 3)`, 검증·테스트 각각 `(69, 3)`이다.


In [ ]:
# 학습 데이터의 섬 목록으로 학습·검증·테스트 데이터 인코딩
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)
train_encoded = encoder.fit_transform(X_train[category_columns])
val_encoded = encoder.transform(X_val[category_columns])
test_encoded = encoder.transform(X_test[category_columns])
encoded_columns = encoder.get_feature_names_out()

# 원래 섬 이름과 인코딩 결과 비교
island_result = pd.DataFrame(train_encoded, columns=encoded_columns, index=X_train.index)
island_result["원래 섬"] = X_train["island"]
display(island_result.head())


### 숫자로 표현하는 이유

문자열을 직접 계산할 수 없는 모델에 섬 정보를 입력하려면 인코딩이 필요하다.

`Biscoe=1`, `Dream=2`, `Torgersen=3`으로 번호를 붙이면 Biscoe와 Dream의 차이는 1, Biscoe와 Torgersen의 차이는 2가 된다.
거리 기반 모델은 Biscoe를 Torgersen보다 Dream에 가깝게 취급하지만, 섬 이름에는 이 순서와 간격의 의미가 없다.
원핫 인코딩은 임의의 순서나 간격 대신 각 범주에 해당하는지를 표현한다.


### 변환 결과 합치기

표준화한 숫자 두 열과 섬을 나타내는 세 열을 합치면 입력 세 열이 숫자 다섯 열로 바뀐다.
행 인덱스를 유지해 각 값이 어느 펭귄의 기록인지 맞춘다.
학습·검증·테스트 데이터의 열 이름과 순서가 같고, 결측값이 남지 않았는지 확인한다.

`isna().sum().sum()`에서 첫 번째 `sum()`은 열별 결측값 수를 구하고, 두 번째 `sum()`은 이를 합쳐 표 전체의 결측값 수를 구한다.


In [ ]:
# 숫자 열과 섬 인코딩 결과를 합쳐 학습·검증·테스트 입력 구성
train_input = pd.DataFrame(train_scaled, columns=numeric_columns, index=X_train.index)
train_input[encoded_columns] = train_encoded
val_input = pd.DataFrame(val_scaled, columns=numeric_columns, index=X_val.index)
val_input[encoded_columns] = val_encoded
test_input = pd.DataFrame(test_scaled, columns=numeric_columns, index=X_test.index)
test_input[encoded_columns] = test_encoded

# 완성된 입력의 크기와 일부 행 확인
print("학습 입력:", train_input.shape)
display(train_input.head())
print("검증 입력:", val_input.shape)
display(val_input.head())
print("테스트 입력:", test_input.shape)
display(test_input.head())

# 열 이름·순서와 남은 결측값 수 확인
print("열 이름과 순서가 같은가:",
      train_input.columns.equals(val_input.columns)
      and train_input.columns.equals(test_input.columns))
print("남은 빈칸 수(학습, 검증, 테스트):",
      train_input.isna().sum().sum(),
      val_input.isna().sum().sum(),
      test_input.isna().sum().sum())


## 파이프라인으로 연결하기


### 파이프라인이 필요한 이유

여러 전처리를 따로 호출하면 중간 결과를 전달하고, 열별 결과를 직접 합쳐야 한다.
새 데이터에도 같은 처리를 반복할 때 단계를 빠뜨리거나 순서를 바꾸기 쉽다.

파이프라인은 처리 단계를 정해진 순서로 연결한 것이다.
각 단계에서 학습한 기준과 처리 순서를 검증·테스트 데이터와 새 데이터에도 일관되게 적용한다.

### Pipeline으로 처리 순서 묶기

`Pipeline`은 여러 처리 도구를 순서대로 연결하는 클래스다. 앞 단계의 변환 결과가 다음 단계의 입력이 된다.
숫자 열은 결측값을 채운 뒤 표준화한다.

각 단계는 `(이름, 처리 도구)`로 지정한다. 객체를 만들 때는 순서만 정하고, `fit` 또는 `fit_transform`을 호출할 때 기준을 학습한다.


In [ ]:
# 숫자 열의 결측값 대체와 표준화를 순서대로 연결
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])


### ColumnTransformer로 열별 처리 연결하기

`ColumnTransformer`는 열별로 지정한 전처리를 적용하고, 결과를 옆으로 합치는 클래스다.
숫자 열에는 결측값 대체와 표준화를, 섬 열에는 원핫 인코딩을 적용한다.

| 입력 | 처리 | 출력 열 수 |
| --- | --- | --- |
| 몸무게·날개 길이 | 중앙값 대체 → 표준화 | 2 |
| 섬 이름 | 원핫 인코딩 | 3 |

`Pipeline`은 처리 순서를, `ColumnTransformer`는 열별 처리를 묶는다.

각 항목은 `(이름, 처리 도구, 열 목록)`으로 지정하며, 변환 결과는 목록에 적은 순서대로 합쳐진다.


In [ ]:
# 숫자 열과 섬 열에 적용할 처리를 하나로 묶기
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), category_columns),
])


### 학습·검증·테스트 데이터 변환하기

학습 데이터로 각 전처리 기준을 정하고 변환한다. 검증·테스트 데이터는 저장된 기준으로 변환한다.
파이프라인으로 묶어도 검증·테스트 데이터에 `fit_transform`을 호출하면 기준을 다시 계산하므로 데이터 누수가 생긴다.

`get_feature_names_out()`으로 최종 열 이름을 확인한다. 이름 앞의 `numeric__`와 `category__`는 해당 열을 만든 처리 묶음이다.


In [ ]:
# 학습 데이터로 기준을 정하고 학습·검증·테스트 데이터를 변환
train_ready = preprocessor.fit_transform(X_train)
val_ready = preprocessor.transform(X_val)
test_ready = preprocessor.transform(X_test)

# 변환 전후 크기와 최종 열 이름 확인
print("학습 입력:", X_train.shape, "→", train_ready.shape)
print("검증 입력:", X_val.shape, "→", val_ready.shape)
print("테스트 입력:", X_test.shape, "→", test_ready.shape)
feature_names = preprocessor.get_feature_names_out()
display(pd.DataFrame(train_ready, columns=feature_names, index=X_train.index).head())


### 개별 전처리와 결과 비교

개별 전처리와 파이프라인은 같은 학습 데이터와 설정을 사용하므로 변환 결과가 같아야 한다.
숫자 두 열과 섬 세 열이 합쳐져 학습 입력은 `(206, 5)`, 검증·테스트 입력은 각각 `(69, 5)`가 된다.

세 비교 결과가 모두 `True`이고 결측값이 0개인지 확인한다.

`np.allclose`는 두 결과의 숫자가 작은 부동소수점 오차를 제외하고 같으면 `True`를 반환한다.


In [ ]:
# 개별 전처리와 파이프라인의 결과 및 남은 결측값 비교
print("학습 결과가 같은가:", np.allclose(train_input, train_ready))
print("검증 결과가 같은가:", np.allclose(val_input, val_ready))
print("테스트 결과가 같은가:", np.allclose(test_input, test_ready))
print("검증 결과에 남은 빈칸 수:", pd.DataFrame(val_ready).isna().sum().sum())
print("테스트 결과에 남은 빈칸 수:", pd.DataFrame(test_ready).isna().sum().sum())


### 새 관측값도 한 번에 변환하기

몸무게가 비어 있는 펭귄 한 마리의 기록을 만들고, 학습 데이터로 기준을 정한 파이프라인으로 변환한다.
`np.nan`은 숫자 데이터의 결측값을 나타낸다.
입력 한 행이 다섯 열로 변환되고, 결측값이 채워졌는지 확인한다.


In [ ]:
# 몸무게가 비어 있는 새 관측값 준비
new_penguin = pd.DataFrame({
    "body_mass_g": [np.nan],
    "flipper_length_mm": [200.],
    "island": ["Dream"],
})

# 저장된 전처리로 새 관측값 변환
new_ready = preprocessor.transform(new_penguin)

# 원본과 변환 결과, 남은 결측값 확인
display(new_penguin)
display(pd.DataFrame(new_ready, columns=feature_names))
print("새 입력의 모양:", new_ready.shape)
print("남은 빈칸 수:", pd.DataFrame(new_ready).isna().sum().sum())


## 펭귄 데이터의 전처리 방법 비교


### 결측값이 있는 행 삭제하기

펭귄 학습 데이터 `X_train`에서 결측값이 있는 행을 삭제하면 관측 수가 줄고, 특정 유형의 관측이 더 많이 빠질 수 있다.

삭제 기준별로 남은 행 수를 비교한다. 숫자 두 열에서 같은 행이 비어 있으므로 전체 열을 기준으로 행을 삭제해도, 몸무게만 기준으로 삭제해도 남는 행은 같다.

| 코드 | 삭제 대상 |
| --- | --- |
| `dropna()` | 결측값이 하나라도 있는 행 |
| `dropna(subset=["body_mass_g"])` | 몸무게가 비어 있는 행 |


In [ ]:
# 전체 열 또는 몸무게를 기준으로 결측 행 삭제
train_complete_rows = X_train.dropna()
train_mass_present = X_train.dropna(subset=["body_mass_g"])

# 삭제 기준별 남은 행과 열 수 비교
deletion_summary = pd.DataFrame(
    [X_train.shape, train_complete_rows.shape, train_mass_present.shape],
    index=["원본", "결측 행 삭제", "몸무게 결측 행 삭제"],
    columns=["행 수", "열 수"],
)
display(deletion_summary)


### 결측값 채우는 방법 비교하기

대체 방법에 따라 결측값에 들어가는 숫자가 달라진다.

| 방법 | 채우는 값 | 고려할 점 |
| --- | --- | --- |
| 평균 | 관측된 숫자의 평균 | 극단값의 영향을 받음 |
| 중앙값 | 크기순으로 가운데 값 | 평균보다 극단값의 영향을 덜 받음 |
| 최빈값 | 가장 자주 등장한 값 | 숫자와 범주에 사용 가능 |
| 고정값 | 지정한 값 | 값의 의미를 데이터에 맞게 정해야 함 |

펭귄 학습 데이터의 같은 몸무게 결측값에 네 방법을 적용해 대체값을 비교한다. 

`strategy`에는 평균 `"mean"`, 중앙값 `"median"`, 최빈값 `"most_frequent"`, 고정값 `"constant"`를 지정한다. 고정값은 `fill_value`로 정한다.

`X_train[["body_mass_g"]]`는 한 열을 2차원 DataFrame으로 선택한다. 변환 결과의 `[:, 0]`은 모든 행에서 첫 열을 꺼내 1차원 배열로 만든다.


In [ ]:
# 비교할 네 가지 결측값 대체 방법 준비
imputation_methods = {
    "평균": SimpleImputer(strategy="mean"),
    "중앙값": SimpleImputer(strategy="median"),
    "최빈값": SimpleImputer(strategy="most_frequent"),
    "고정값": SimpleImputer(strategy="constant", fill_value=4000),
}

# 같은 학습 데이터에 각 방법을 적용해 결과 모으기
imputation_comparison = pd.DataFrame({"원래 몸무게": X_train["body_mass_g"]})
for name, method in imputation_methods.items():
    filled_mass = method.fit_transform(X_train[["body_mass_g"]])
    imputation_comparison[name] = filled_mass[:, 0]

# 몸무게가 비어 있던 행의 대체값 비교
display(imputation_comparison.loc[missing_rows].round(2))


### 스케일링 방법 비교

스케일링은 숫자의 크기나 범위를 조절하는 작업이다. 전처리 클래스마다 기준이 다르다.

| 클래스 | 변환 방법 | 특징 |
| --- | --- | --- |
| `StandardScaler` | `(값 - 평균) / 표준편차` | 학습 데이터의 평균을 0, 표준편차를 1로 맞춘다. |
| `MinMaxScaler` | `(값 - 최솟값) / (최댓값 - 최솟값)` | 학습 데이터의 최솟값을 0, 최댓값을 1로 맞춘다. |
| `RobustScaler` | `(값 - 중앙값) / IQR` | 중앙값과 IQR을 사용해 극단값의 영향을 덜 받는 기준으로 변환한다. |

IQR은 75% 지점의 값에서 25% 지점의 값을 뺀 것으로, 가운데 50%가 퍼진 범위다.

펭귄 학습 데이터의 결측값을 중앙값으로 채운 `train_filled`에 세 방법을 적용한다. 같은 몸무게가 각 방법에서 어떤 값으로 바뀌는지 비교한다.


In [ ]:
# 비교할 스케일링 도구 준비
scaling_methods = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

# 각 도구로 학습·검증 데이터 변환
scaled_train_results = {}
scaled_val_results = {}
for name, method in scaling_methods.items():
    scaled_train_results[name] = method.fit_transform(train_filled)
    scaled_val_results[name] = method.transform(val_filled)

# 같은 몸무게의 변환 결과 비교
scaling_comparison = pd.DataFrame({"원래 몸무게": train_filled[:, 0]}, index=X_train.index)
for name, result in scaled_train_results.items():
    scaling_comparison[name] = result[:, 0]
display(scaling_comparison.head().round(3))


## 추가 전처리


### 가상 구매 기록 만들기

한 행이 상품 구매 한 건을 나타내는 가상 데이터다. 같은 구매가 반복 저장된 기록을 포함해 13행이다.

| 열 | 의미 |
| --- | --- |
| `purchase_id` | 구매 기록의 식별 번호 |
| `store` | 구매한 지점 이름 |
| `amount` | 문자열로 기록된 구매 금액 |
| `amount_unit` | 금액 단위인 원 또는 천원 |

`head()`로 처음 5행의 기록을 확인한다.


In [ ]:
# 표기와 단위가 섞인 가상 구매 기록 준비
raw_sales = pd.DataFrame({
    "purchase_id": ["P001", "P001", "P002", "P003", "P004", "P005", "P006",
                    "P007", "P008", "P009", "P010", "P011", "P012"],
    "store": [" Gangnam ", " Gangnam ", "HONGDAE", "gangnam", " Hongdae ", "JAMSIL",
              "jamsil", "GANGNAM", "hongdae", "Jamsil", "gangnam", "HONGDAE", "Jamsil"],
    "amount": ["10", "10", "12000", "15", "18000", "20", "22000",
               "25", "28000", "30", "200", "-5", "unknown"],
    "amount_unit": ["천원", "천원", "원", "천원", "원", "천원", "원",
                    "천원", "원", "천원", "천원", "천원", "원"],
})
print("전체 행과 열:", raw_sales.shape)
display(raw_sales.head())


### 중복 기록 제거하기

같은 구매가 여러 번 저장되면 구매 건수와 금액이 과대 집계된다.
구매 번호와 기록 내용을 함께 확인한다.

`duplicated()`는 모든 열이 같은 행 중 두 번째 등장부터 `True`로 표시한다.
`drop_duplicates()`는 중복 중 첫 행을 남긴 표를 반환한다.


In [ ]:
# 같은 구매가 반복 저장된 행 확인
duplicate_rows = raw_sales.duplicated()
display(raw_sales.loc[duplicate_rows])

# 중복을 제거한 작업용 데이터 생성
clean_sales = raw_sales.drop_duplicates().copy()
print("정리 전후 행 수:", len(raw_sales), "→", len(clean_sales))


### 문자 표기 통일하기

`" Gangnam "`과 `"gangnam"`처럼 공백이나 대소문자가 다르면 같은 지점도 다른 범주로 취급될 수 있다.
`str.strip()`으로 앞뒤 공백을 제거하고, `str.title()`로 단어의 첫 글자만 대문자로 맞춘다.


In [ ]:
# 원래 지점 이름을 보관하고 공백·대소문자 통일
store_before = clean_sales["store"].copy()
clean_sales["store"] = clean_sales["store"].str.strip().str.title()

# 지점 이름의 정리 전후 비교
display(pd.DataFrame({"정리 전": store_before, "정리 후": clean_sales["store"]}))


### 구매 금액 정리하기

문자열 금액을 숫자로 바꾸고, 천원 단위의 값에 1,000을 곱해 원으로 통일한다.
이 예제에서 음수 금액은 입력 오류이며, 원래 금액을 알 수 없어 결측값으로 처리한다.

`pd.to_numeric(..., errors="coerce")`는 숫자로 바꿀 수 없는 `"unknown"`을 결측값으로 만든다.
`map()`은 각 단위에 해당하는 변환 배수를 가져온다. 정리한 원 단위 금액은 `amount_won`에 저장한다.


In [ ]:
# 금액 문자열을 숫자로 바꾸고 원 단위로 통일
amount_values = pd.to_numeric(clean_sales["amount"], errors="coerce")
unit_factors = clean_sales["amount_unit"].map({"원": 1, "천원": 1000})
clean_sales["amount_won"] = amount_values * unit_factors

# 음수로 기록된 금액을 확인하고 결측값으로 변경
invalid_amount = clean_sales["amount_won"] < 0
display(clean_sales.loc[invalid_amount, ["purchase_id", "amount", "amount_unit", "amount_won"]])
clean_sales.loc[invalid_amount, "amount_won"] = np.nan

# 정리된 구매 기록과 남은 결측값 수 확인
display(clean_sales[["purchase_id", "store", "amount", "amount_unit", "amount_won"]])
print("금액 결측값 수:", clean_sales["amount_won"].isna().sum())


### 이상치 후보 확인하기

이상치는 다른 관측값과 비교해 유난히 크거나 작은 값이다. 입력 오류인지, 실제로 드문 관측인지 원인을 확인한 뒤 처리한다.
값을 정렬했을 때 25% 지점을 `Q1`, 75% 지점을 `Q3`라고 한다. 두 값의 차이인 IQR은 가운데 50%가 퍼진 범위다.
`Q1 - 1.5 × IQR`보다 작거나 `Q3 + 1.5 × IQR`보다 큰 값을 확인 대상으로 표시할 수 있다.

정리한 구매 기록에서 금액이 있는 행만 선택해 경계를 계산하고, 경계를 벗어난 기록을 표시한다.
`quantile(0.25)`와 `quantile(0.75)`는 각각 25%, 75% 지점의 값을 구한다.


In [ ]:
# 금액이 있는 구매 기록으로 이상치 확인 경계 계산
amount_check = clean_sales[["purchase_id", "amount_won"]].dropna().copy()
q1 = amount_check["amount_won"].quantile(0.25)
q3 = amount_check["amount_won"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print("확인 경계:", lower_bound, "~", upper_bound)

# 경계를 벗어난 구매 기록을 확인 대상으로 표시
amount_check["확인 필요"] = (
    (amount_check["amount_won"] < lower_bound)
    | (amount_check["amount_won"] > upper_bound)
)
display(amount_check)


### 이상치를 확인한 뒤 처리하기

| 확인 결과 | 처리 방법 |
| --- | --- |
| 단위나 입력 오류이며 원래 값을 확인할 수 있음 | 올바른 값으로 수정 |
| 오류는 확인했지만 원래 값을 알 수 없음 | 결측값으로 바꾼 뒤 대체하거나, 분석 목적에 따라 행 제외 |
| 실제로 관측된 큰 값 | 값을 유지하고 분석 목적에 맞는 변환 검토 |

경계를 벗어났다는 이유만으로 기록을 삭제하지 않는다.
이 예제에서는 200,000원이 실제 구매 금액이라고 가정하고 유지한다.


### 로그 변환으로 큰 값의 격차 줄이기

구매 금액처럼 대부분은 작고 일부 값만 매우 큰 데이터에는 로그 변환을 고려할 수 있다.
로그 변환은 값이 커질수록 증가 폭을 줄여 큰 값 사이의 격차를 압축한다.

`np.log1p(x)`는 자연로그 `log(1 + x)`를 계산한다.
금액이 확인된 구매 기록에 적용하고, 원 단위 금액과 변환값을 비교한다.


In [ ]:
# 금액이 확인된 구매 기록에 로그 변환 적용
log_result = amount_check[["purchase_id", "amount_won"]].copy()
log_result["log_amount"] = np.log1p(log_result["amount_won"])

# 원래 구매 금액과 로그 변환값 비교
display(log_result.round(3))


### 로그 변환 결과 해석

200,000원인 구매 기록도 남아 있지만, 로그 변환 후에는 다른 금액과의 숫자 격차가 줄어든다.
변환값은 원 단위 금액이 아니라 로그 값이며, 구매 금액의 크기 순서는 유지된다.


## 전처리 실습과 점검


### Auto MPG 데이터

자동차의 무게·마력·제조 지역으로 연비를 예측하는 문제의 입력을 준비한다.
Auto MPG는 자동차의 특성과 연비를 담은 398행의 데이터다. 한 행은 자동차 한 대의 기록이다.

| 역할 | 열 | 의미와 단위 |
| --- | --- | --- |
| 숫자 입력 | `weight` | 차량 무게, lb(파운드) |
| 숫자 입력 | `horsepower` | 엔진 마력 |
| 범주 입력 | `origin` | 제조 지역: usa·europe·japan |
| 정답 | `mpg` | 미국 갤런당 주행 마일 수 |

`sns.load_dataset("mpg")`로 데이터를 불러온다. 입력의 일부 행과 결측값 개수, 결측값이 있는 기록을 확인한다.


In [ ]:
# 자동차 데이터를 불러오고 전처리할 입력 선택
mpg = sns.load_dataset("mpg")
exercise_numeric_columns = ["weight", "horsepower"]
exercise_category_columns = ["origin"]
exercise_X = mpg[exercise_numeric_columns + exercise_category_columns].copy()

# 입력의 일부 행과 결측값이 있는 기록 확인
display(exercise_X.head())
display(exercise_X.isna().sum())
display(exercise_X[exercise_X.isna().any(axis=1)])


### 직접 작성하기: 자동차 데이터 전처리

- 자동차 입력을 학습 60%, 검증 20%, 테스트 20%로 나눈다.
- 숫자 열의 결측값을 학습 데이터의 중앙값으로 채운다.
- 숫자 열을 표준화한다.
- 제조 지역을 원핫 인코딩해 숫자로 표현한다.
- 전처리 과정을 파이프라인으로 묶고, 학습에서 정한 기준으로 검증·테스트 데이터를 변환한다.

변환 결과에 결측값이 없는지, 학습 숫자 열의 평균이 약 0이고 표준편차가 약 1인지 확인한다.
